In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

In [2]:
url = "https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics/player-statistics"

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

driver.maximize_window()
driver.get(url)
time.sleep(5)

In [3]:
print(driver.title)

Player Stats | FIFA World Cup 2026™


In [4]:
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(3)

In [5]:
print(driver.page_source[:1000])

<html lang="en" dir="ltr" data-react-helmet="lang,dir"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><meta name="theme-color" content="#020F2A"><meta name="msapplication-TileImage" content="/mstile-150x150.png?v=6b9bed24d1df59ca113a3828909d3924"><meta name="msapplication-TileColor" content="#326295"><meta name="google-site-verification" content="boCUJZPlju606sys-ZcnqZAThCMVJWEvwc6JXvElJTE"><link rel="icon" href="/favicon.ico?v=4c4914f90c578869e7375b03cf029202"><link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"><link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png?v=1ea068c804e8ba88b84f6e9598e3172d"><link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png?v=7ee355e68b687435c3e5f74e2e831325"><link rel="apple-touch-icon" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"><link rel="manifest" href="/manifest.webmanifest?v=82b16ef736854fe

In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(driver.page_source, "html.parser")

print(len(soup.find_all("table")))
print(len(soup.find_all("tr")))

1
51


In [7]:
table = soup.find("table")

rows = table.find_all("tr")
print("Total Rows:", len(rows))

Total Rows: 51


In [8]:
print(rows[1].prettify())

<tr class="row-even">
 <td class="sticky-column rank-column">
  <div class="ranking-cell sm">
   <span class="ranking-value">
    1
   </span>
  </div>
 </td>
 <td class="sticky-column list-cell-column">
  <div class="list-cell">
   <div class="avatar avatar--player sm">
    <img alt="" src="https://digitalhub.fifa.com/transform/66f6087d-9563-4644-8f10-5614ef6e1e51/MBAPPE-Kylian_389867?&amp;io=transform:crop,height:850,width:850&amp;quality=75"/>
   </div>
   <div class="content-container">
    <div class="main-text">
     Kylian Mbappe
    </div>
    <div class="extra-info-container">
     <p class="extra-info-description extra-info-description--iconed">
      <div class="extra-info-image">
       <img alt="FRA" class="image" src="https://api.fifa.com/api/v3/picture/flags-sq-3/FRA"/>
      </div>
      <span class="dsk-description">
       FRA
      </span>
      <span class="mob-description">
       FRA
      </span>
     </p>
     <p class="extra-info-description">
      <span class

In [9]:
players = []

for row in rows[1:]:
    cols = row.find_all("td")

    if len(cols) >= 5:
        rank = cols[0].get_text(strip=True)

        name = cols[1].find("div", class_="main-text").get_text(strip=True)

        spans = cols[1].find_all("span")
        country = spans[0].get_text(strip=True)
        position = spans[2].get_text(strip=True)

        matches = cols[2].get_text(strip=True)
        goals = cols[3].get_text(strip=True)
        minutes = cols[4].get_text(strip=True)

        players.append({
            "Rank": rank,
            "Player": name,
            "Country": country,
            "Position": position,
            "Matches": matches,
            "Goals": goals,
            "Minutes": minutes
        })

In [10]:
df = pd.DataFrame(players)

df.head()

,Rank,Player,Country,Position,Matches,Goals,Minutes
0,1,Kylian Mbappe,FRA,FW,10,4,769
1,2,Lionel Messi,ARG,FW,8,4,853
2,3,Jude Bellingham,ENG,MF,7,1,698
3,4,Erling Haaland,NOR,FW,7,0,537
4,5,Ousmane Dembele,FRA,FW,6,2,648


In [11]:
df.to_csv("../data/players_raw.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [12]:
from selenium.webdriver.common.by import By

tabs = driver.find_elements(By.TAG_NAME, "button")

for i, tab in enumerate(tabs):
    text = tab.text.strip()
    if text:
        print(i, text)

1 English
11 TEAMS & STATS
12 LATEST
13 FANTASY & GAMING
14 MORE
16 adidas Golden Boot
17 Attacking
18 Distribution
19 Defending
20 Discipline
21 Goalkeeping
22 Movement
23 Physical
25 All Teams
26 Glossary
27 Goals
28 Assists
29 Minutes Played
30 Load more
31 Preference Center
32 Reject All
33 I'm OK with that


In [13]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

categories = [
    "Attacking",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping",
    "Movement",
    "Physical"
]

for category in categories:
    button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, f"//button[normalize-space()='{category}']"))
    )

    driver.execute_script("arguments[0].click();", button)
    time.sleep(3)

    print(f"Opened: {category}")

Opened: Attacking
Opened: Distribution
Opened: Defending
Opened: Discipline
Opened: Goalkeeping
Opened: Movement
Opened: Physical
